Features: 
| Feature | Meaning |
|---|---|
| `cohort_month` | Month when the users/subscriptions started |
| `months_since_signup` | Customer age in months, such as Month 0, Month 1, Month 2 |
| `cohort_size` | Original number of users/subscriptions in that cohort |
| `retained_users` | Number of users/subscriptions still active after N months |
| `retention_rate` | `retained_users / cohort_size` |
| `product_id` | Product linked to the subscription, if available |
| `plan_id` | Plan linked to the subscription, if available |
| `country` | User/customer country, if available |
| `acquisition_channel` | How the user was acquired, if available |
| `saved_at` | Timestamp when the table was saved |


The cohort table is about tracking how well each starting group of users/subscriptions stays active over time.


In [0]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number, to_timestamp, lit, create_map, upper, trim
from pyspark.sql.types import DecimalType
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from functools import reduce
from itertools import chain
import sklearn
from pyspark.sql.types import DoubleType, FloatType, DecimalType
print(pyspark.__version__)


In [0]:
builder = (
    SparkSession.builder
    .appName("user_bronze")
)
spark = builder.getOrCreate()
silver_db = "/Volumes/datalake_catalog/datalake_schema/silver/"
snapshot_date = F.current_date()
cutoff_date = F.lit("2026-04-30").cast("date")
users = (
    spark.read
    .format("delta")
    .load(silver_db+"/users")
)
products = (
    spark.read
    .format("delta")
    .load(silver_db+"/products")
)
plans = (
    spark.read
    .format("delta")
    .load(silver_db+"/plans")
)
subs = (
    spark.read
    .format("delta")
    .load(silver_db+"/subscriptions")
)

changes = (
    spark.read
    .format("delta")
    .load(silver_db+"/subscription_changes")
)
payments = (
    spark.read
    .format("delta")
    .load(silver_db+"/payments")
)
licenses = (
    spark.read
    .format("delta")
    .load(silver_db+"/licenses")
)
allocations = (
    spark.read
    .format("delta")
    .load(silver_db+"/license_allocations")
)
usage = (
    spark.read
    .format("delta")
    .load(silver_db+"/usage_events")
)
tickets = (
    spark.read
    .format("delta")
    .load(silver_db+"/support_tickets")
)

## cohort_month

In [0]:
df1 = subs.groupBy("user_id").agg(
    F.min("start_date").alias("first_subscription_start_date")
).withColumn(
    "cohort_month",
    F.date_trunc("month", F.col("first_subscription_start_date"))
)
df1.count()

In [0]:
display(df1)

In [0]:
payment_activity = payments.filter(F.col("payment_status") == "success") \
    .join(
        subs.select("subscription_id", "user_id", "plan_id"),
        on="subscription_id",
        how="left"
    ) \
    .withColumn(
        "activity_month",
        F.date_trunc("month", F.col("payment_date"))
    ) \
    .select("user_id", "activity_month", "subscription_id", "plan_id") \
    .dropDuplicates()
df2 = df1.join(
    payment_activity,
    on="user_id",
    how="inner"
).withColumn(
    "months_since_signup",
    (
        (F.year("activity_month") - F.year("cohort_month")) * 12
        + (F.month("activity_month") - F.month("cohort_month"))
    )
).filter(
    (F.col("months_since_signup") >= 0) &
    (F.col("months_since_signup") <= 12)
)

## product_id, plan_id


In [0]:
df3 = df2.join(
    plans.select("plan_id", "product_id"),
    on="plan_id",
    how="left"
)
display(df3)

## country

In [0]:
df4 = df3.join(
    users.select("user_id", "country", "acquisition_channel"),
    on="user_id",
    how="left"
)
display(df4)

## Multiple cohort tables:

1. cohort_retention_overall
   groupBy(cohort_month, months_since_signup)

2. cohort_retention_by_product
   groupBy(cohort_month, months_since_signup, product_id)

3. cohort_retention_by_plan
   groupBy(cohort_month, months_since_signup, plan_id)

4. cohort_retention_by_country
   groupBy(cohort_month, months_since_signup, country)

5. cohort_retention_by_acquisition_channel
   groupBy(cohort_month, months_since_signup, acquisition_channel)

## 1. Overall

In [0]:
# overall
cohort_size_overall = df4.select(
    "cohort_month",
    "user_id"
).dropDuplicates().groupBy(
    "cohort_month"
).agg(
    F.count("user_id").alias("cohort_size")
)

retained_users_overall = df4.select(
    "cohort_month",
    "months_since_signup",
    "user_id"
).dropDuplicates().groupBy(
    "cohort_month",
    "months_since_signup"
).agg(
    F.count("user_id").alias("retained_users")
)

cohort_retention_overall = retained_users_overall.join(
    cohort_size_overall,
    on="cohort_month",
    how="left"
).withColumn(
    "retention_rate",
    F.round(F.col("retained_users") / F.col("cohort_size"), 2)
).orderBy(
    "cohort_month",
    "months_since_signup"
).withColumn(
    "created_at",
    F.current_timestamp()
)

display(cohort_retention_overall)

## 2. Product_id

In [0]:
# product_id
cohort_size_product = df4.select(
    "cohort_month",
    "product_id",
    "user_id"
).dropDuplicates().groupBy(
    "cohort_month",
    "product_id"
).agg(
    F.count("user_id").alias("cohort_size")
)

retained_users_product = df4.select(
    "cohort_month",
    "months_since_signup",
    "product_id",
    "user_id"
).dropDuplicates().groupBy(
    "cohort_month",
    "months_since_signup",
    "product_id"
).agg(
    F.count("user_id").alias("retained_users")
)

cohort_retention_by_product = retained_users_product.join(
    cohort_size_product,
    on=["cohort_month", "product_id"],
    how="left"
).withColumn(
    "retention_rate",
    F.round(F.col("retained_users") / F.col("cohort_size"), 2)
).withColumn(
    "created_at",
    F.current_timestamp()
)
display(cohort_retention_by_product)

## 3. plan_id

In [0]:
cohort_size_plan = df4.select(
    "cohort_month",
    "plan_id",
    "user_id"
).dropDuplicates().groupBy(
    "cohort_month",
    "plan_id"
).agg(
    F.count("user_id").alias("cohort_size")
)

retained_users_plan = df4.select(
    "cohort_month",
    "months_since_signup",
    "plan_id",
    "user_id"
).dropDuplicates().groupBy(
    "cohort_month",
    "months_since_signup",
    "plan_id"
).agg(
    F.count("user_id").alias("retained_users")
)

cohort_retention_by_plan = retained_users_plan.join(
    cohort_size_plan,
    on=["cohort_month", "plan_id"],
    how="left"
).withColumn(
    "retention_rate",
    F.round(F.col("retained_users") / F.col("cohort_size"), 2)
).withColumn(
    "created_at",
    F.current_timestamp()
)
display(cohort_retention_by_plan)

## 4. acquisition_channel

In [0]:
cohort_size_channel = df4.select(
    "cohort_month",
    "acquisition_channel",
    "user_id"
).dropDuplicates().groupBy(
    "cohort_month",
    "acquisition_channel"
).agg(
    F.count("user_id").alias("cohort_size")
)

retained_users_channel = df4.select(
    "cohort_month",
    "months_since_signup",
    "acquisition_channel",
    "user_id"
).dropDuplicates().groupBy(
    "cohort_month",
    "months_since_signup",
    "acquisition_channel"
).agg(
    F.count("user_id").alias("retained_users")
)

cohort_retention_by_acquisition_channel = retained_users_channel.join(
    cohort_size_channel,
    on=["cohort_month", "acquisition_channel"],
    how="left"
).withColumn(
    "retention_rate",
    F.round(F.col("retained_users") / F.col("cohort_size"), 2)
).withColumn(
    "created_at",
    F.current_timestamp()
)
display(retained_users_channel)

## Country

In [0]:
cohort_size_country = df4.select(
    "cohort_month",
    "country",
    "user_id"
).dropDuplicates().groupBy(
    "cohort_month",
    "country"
).agg(
    F.count("user_id").alias("cohort_size")
)

retained_users_country = df4.select(
    "cohort_month",
    "months_since_signup",
    "country",
    "user_id"
).dropDuplicates().groupBy(
    "cohort_month",
    "months_since_signup",
    "country"
).agg(
    F.count("user_id").alias("retained_users")
)

cohort_retention_by_country = retained_users_country.join(
    cohort_size_country,
    on=["cohort_month", "country"],
    how="left"
).withColumn(
    "retention_rate",
    F.round(F.col("retained_users") / F.col("cohort_size"), 2)
).withColumn(
    "created_at",
    F.current_timestamp()
)
display(cohort_retention_by_country)

# Saving

If same cohort_month + months_since_signup exists:
    replace/update the row

If it does not exist:
    insert new row

In [0]:
def quote_table_name(table_name):
    return ".".join([f"`{part}`" for part in table_name.split(".")])


def get_table_latest_created_at(table_name):
    quoted_table = quote_table_name(table_name)

    details = spark.sql(f"DESCRIBE DETAIL {quoted_table}").collect()[0]
    properties = details["properties"] or {}

    return properties.get("latest_created_at")


def set_table_latest_created_at(table_name, latest_created_at):
    quoted_table = quote_table_name(table_name)

    spark.sql(f"""
        ALTER TABLE {quoted_table}
        SET TBLPROPERTIES (
            'latest_created_at' = '{latest_created_at}'
        )
    """)


def upsert_delta_table(df, table_name, merge_keys):
    """
    Table-level versioned upsert.

    Rules:
    - If table does not exist: save whole dataframe.
    - If table exists:
        - get max(created_at) from current dataframe
        - compare with saved table metadata latest_created_at
        - if current max is not newer: skip
        - if current max is newer: upsert whole dataframe
    """

    if "created_at" not in df.columns:
        raise ValueError("created_at column is required")

    current_latest_created_at = (
        df
        .select(F.max(F.col("created_at")).alias("latest_created_at"))
        .collect()[0]["latest_created_at"]
    )

    if current_latest_created_at is None:
        print(f"Skipped table: {table_name}")
        print("Reason: dataframe has no created_at value")
        print("Inserted rows: 0")
        print("Updated rows: 0")
        return

    current_latest_created_at_str = str(current_latest_created_at)

    if not spark.catalog.tableExists(table_name):
        df.write.format("delta") \
            .mode("overwrite") \
            .saveAsTable(table_name)

        set_table_latest_created_at(table_name, current_latest_created_at_str)

        print(f"Created new table: {table_name}")
        print(f"Inserted rows: {df.count()}")
        print("Updated rows: 0")
        print(f"latest_created_at: {current_latest_created_at_str}")
        return

    saved_latest_created_at = get_table_latest_created_at(table_name)

    if saved_latest_created_at is not None:
        should_skip = spark.sql(f"""
            SELECT
                to_timestamp('{current_latest_created_at_str}')
                <= to_timestamp('{saved_latest_created_at}')
                AS should_skip
        """).collect()[0]["should_skip"]

        if should_skip:
            print(f"Skipped table: {table_name}")
            print(f"Reason: current dataframe is not newer than saved table")
            print(f"Current latest created_at: {current_latest_created_at_str}")
            print(f"Saved latest created_at: {saved_latest_created_at}")
            print("Inserted rows: 0")
            print("Updated rows: 0")
            return

    target = DeltaTable.forName(spark, table_name)

    merge_condition = " AND ".join([
        f"target.`{key}` <=> source.`{key}`"
        for key in merge_keys
    ])

    update_set = {
        col: f"source.`{col}`"
        for col in df.columns
    }

    insert_set = {
        col: f"source.`{col}`"
        for col in df.columns
    }

    target.alias("target") \
        .merge(
            df.alias("source"),
            merge_condition
        ) \
        .whenMatchedUpdate(
            condition="to_timestamp(source.`created_at`) > to_timestamp(target.`created_at`)",
            set=update_set
        ) \
        .whenNotMatchedInsert(
            values=insert_set
        ) \
        .execute()

    set_table_latest_created_at(table_name, current_latest_created_at_str)

    history = spark.sql(f"DESCRIBE HISTORY {quote_table_name(table_name)} LIMIT 1")
    metrics = history.select("operationMetrics").collect()[0]["operationMetrics"]

    inserted_rows = int(metrics.get("numTargetRowsInserted", 0))
    updated_rows = int(metrics.get("numTargetRowsUpdated", 0))

    print(f"Upserted table: {table_name}")
    print(f"Inserted rows: {inserted_rows}")
    print(f"Updated rows: {updated_rows}")
    print(f"latest_created_at: {current_latest_created_at_str}")


In [0]:
overall_keys = [
    "cohort_month",
    "months_since_signup"
]
upsert_delta_table(
    cohort_retention_overall,
    "datalake_catalog.gold.gold_cohort_retention_overall",
    overall_keys
)
product_keys = [
    "cohort_month",
    "months_since_signup",
    "product_id"
]

upsert_delta_table(
    cohort_retention_by_product,
    "datalake_catalog.gold.gold_cohort_retention_by_product",
    product_keys
)
plan_keys = [
    "cohort_month",
    "months_since_signup",
    "plan_id"
]

upsert_delta_table(
    cohort_retention_by_plan,
    "datalake_catalog.gold.gold_cohort_retention_by_plan",
    plan_keys
)
country_keys = [
    "cohort_month",
    "months_since_signup",
    "country"
]

upsert_delta_table(
    cohort_retention_by_country,
    "datalake_catalog.gold.gold_cohort_retention_by_country",
    country_keys
)
channel_keys = [
    "cohort_month",
    "months_since_signup",
    "acquisition_channel"
]

upsert_delta_table(
    cohort_retention_by_acquisition_channel,
    "datalake_catalog.gold.gold_cohort_retention_by_acquisition_channel",
    channel_keys
)

In [0]:
overall_keys = [
    "cohort_month",
    "months_since_signup"
]
upsert_delta_table(
    cohort_retention_overall,
    "datalake_catalog.gold.gold_cohort_retention_overall",
    overall_keys
)

In [0]:
# Checking the upsert function
table_name = "datalake_catalog.gold.test_gold_cohort_retention_sim"
merge_keys = ["cohort_month", "months_since_signup"]

columns = [
    "cohort_month",
    "months_since_signup",
    "retained_users",
    "cohort_size",
    "retention_rate",
    "created_at",
]


def make_retention_df(data):
    return (
        spark.createDataFrame(data, columns)
        .withColumn("cohort_month", F.to_timestamp("cohort_month"))
        .withColumn("created_at", F.to_timestamp("created_at"))
    )


old_data = [
    ("2024-01-01 00:00:00", 0, 120, 300, 0.40, "2026-05-03 04:00:00.000000"),
    ("2024-01-01 00:00:00", 1, 96,  300, 0.32, "2026-05-03 04:00:00.000000"),
    ("2024-01-01 00:00:00", 2, 75,  300, 0.25, "2026-05-03 04:00:00.000000"),
    ("2024-02-01 00:00:00", 0, 180, 420, 0.43, "2026-05-03 04:00:00.000000"),
]

same_created_at_data = [
    ("2024-01-01 00:00:00", 0, 999, 300, 3.33, "2026-05-03 04:00:00.000000"),
    ("2024-01-01 00:00:00", 1, 888, 300, 2.96, "2026-05-03 04:00:00.000000"),
    ("2024-01-01 00:00:00", 2, 777, 300, 2.59, "2026-05-03 04:00:00.000000"),
    ("2024-02-01 00:00:00", 0, 666, 420, 1.59, "2026-05-03 04:00:00.000000"),
]

older_data = [
    ("2024-01-01 00:00:00", 0, 130, 300, 0.43, "2026-05-03 03:30:00.000000"),
    ("2024-01-01 00:00:00", 1, 90,  300, 0.30, "2026-05-03 03:30:00.000000"),
    ("2024-01-01 00:00:00", 2, 70,  300, 0.23, "2026-05-03 03:30:00.000000"),
    ("2024-02-01 00:00:00", 0, 170, 420, 0.40, "2026-05-03 03:30:00.000000"),
]

newer_update_data = [
    ("2024-01-01 00:00:00", 0, 150, 300, 0.50, "2026-05-03 05:00:00.000000"),
    ("2024-01-01 00:00:00", 1, 123, 300, 0.41, "2026-05-03 05:00:00.000000"),
    ("2024-01-01 00:00:00", 2, 102, 300, 0.34, "2026-05-03 05:00:00.000000"),
    ("2024-02-01 00:00:00", 0, 210, 420, 0.50, "2026-05-03 05:00:00.000000"),
]

newer_mixed_data = [
    ("2024-01-01 00:00:00", 0, 160, 300, 0.53, "2026-05-03 06:00:00.000000"),
    ("2024-01-01 00:00:00", 1, 132, 300, 0.44, "2026-05-03 06:00:00.000000"),
    ("2024-03-01 00:00:00", 0, 240, 500, 0.48, "2026-05-03 06:00:00.000000"),
    ("2024-03-01 00:00:00", 1, 205, 500, 0.41, "2026-05-03 06:00:00.000000"),
]


df_old = make_retention_df(old_data)
df_same_created_at = make_retention_df(same_created_at_data)
df_older = make_retention_df(older_data)
df_newer_update = make_retention_df(newer_update_data)
df_newer_mixed = make_retention_df(newer_mixed_data)


# clean test table before starting.
# spark.sql(f"DROP TABLE IF EXISTS {table_name}")


print("TEST 1: create old table")
upsert_delta_table(df_old, table_name, merge_keys)

print("TEST 2: same created_at, same keys, different values should skip")
upsert_delta_table(df_same_created_at, table_name, merge_keys)

print("TEST 3: older created_at should skip")
upsert_delta_table(df_older, table_name, merge_keys)

print("TEST 4: newer created_at, same keys should update")
upsert_delta_table(df_newer_update, table_name, merge_keys)

print("TEST 5: newer created_at, mixed existing and new keys should update and insert")
upsert_delta_table(df_newer_mixed, table_name, merge_keys)

print("FINAL TABLE")
spark.table(table_name) \
    .orderBy("cohort_month", "months_since_signup") \
    .show(truncate=False)

print("TABLE PROPERTIES")
spark.sql(f"DESCRIBE DETAIL {table_name}") \
    .select("properties") \
    .show(truncate=False)


In [0]:
# gold_tables = [
#     "datalake_catalog.gold.gold_cohort_retention_overall",
#     "datalake_catalog.gold.gold_cohort_retention_by_product",
#     "datalake_catalog.gold.gold_cohort_retention_by_plan",
#     "datalake_catalog.gold.gold_cohort_retention_by_country",
#     "datalake_catalog.gold.gold_cohort_retention_by_acquisition_channel"
# ]

# for table_name in gold_tables:
#     spark.sql(f"DROP TABLE IF EXISTS {table_name}")
#     print(f"Dropped table if existed: {table_name}")


retained_users
retention_rate